# Bazaario Analytics — Notebook 1: Supervised Learning
## IS215 Digital Business — Group 4

### What this notebook does

This notebook trains **5 separate ML models**, each answering a different business question.

| # | Business Question | Type | Model | What the metric means |
|---|------------------|------|-------|--------|
| 1 | Will a customer pay cashless or cash? | Classification | Logistic Regression | % of correct predictions |
| 2 | How much revenue will a vendor earn per day? | Regression | Linear Regression | How close predictions are to actual revenue |
| 3 | How many orders per hour will the platform get? | Regression | Linear Regression | How close predictions are to actual orders |
| 4 | Is a customer review positive or negative? | Classification | Keyword NLP | % of reviews correctly classified |
| 5 | What is each vendor's total revenue? | Regression | Linear Regression | How close predictions are to actual revenue |

Each model is trained and evaluated independently — they don't share metrics.

**Technical note:** Classification uses Logistic Regression (predicts a category: yes/no). Regression uses Linear Regression (predicts a number: dollars/orders). Both are supervised learning — the model learns from labelled historical data.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import RFE
from sklearn.model_selection import train_test_split, GridSearchCV
import matplotlib.pyplot as plt

def preprocess(X):
    """
    Prepare data for machine learning.
    
    Plain English: Scale all numbers to a 0-1 range so no single column
    dominates, and convert text categories into numeric 0/1 columns.
    
    Technical: MinMaxScaler normalisation + pandas get_dummies (dummy encoding
    with drop_first to avoid multicollinearity).
    """
    scaler = MinMaxScaler(feature_range=(0, 1))
    for col in X.select_dtypes(include=[np.number]).columns:
        X[col] = scaler.fit_transform(X[[col]])
    cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()
    if cat_cols:
        X = pd.get_dummies(X, columns=cat_cols, drop_first=True, dtype=int)
    return X, scaler

# Load data
txn = pd.read_csv('data/transactions.csv')
reviews = pd.read_csv('data/reviews.csv')
txn['date'] = pd.to_datetime(txn['date'])
print(f"Loaded {len(txn):,} transactions, {len(reviews)} reviews")

---
# Model 1: Cashless Payment Prediction

**Business question:** Given a transaction's amount, time, vendor category, and region — will the customer pay cashless or cash?

**Who benefits:** Admin can understand which customer segments prefer cashless and target promotions accordingly.

**How it works (plain English):** We show the computer thousands of past transactions where we know the payment method. It finds patterns like "weekend + higher amount = more likely cashless." Then we test it on transactions it hasn't seen to check if the patterns hold.

**Technical:** Logistic Regression classifier. Binary classification (yes/no). Evaluated with accuracy, confusion matrix, classification report (precision, recall, F1). Feature selection via RFE. Hyperparameter tuning via GridSearchCV with 5-fold cross-validation.

In [ ]:
# Create target: cashless (yes) vs cash (no)
txn['is_cashless'] = txn['payment_method'].apply(lambda x: 'yes' if x in ['wallet', 'qr_code'] else 'no')

# Features that might influence payment choice
X1 = txn[['amount', 'hour', 'is_weekend', 'vendor_category', 'region']].copy()
y1 = txn['is_cashless']
X1, _ = preprocess(X1)

# Split 80% for training (model learns), 20% for testing (we evaluate)
# Technical: stratify=y1 keeps the same cashless/cash ratio in both sets
X_tr, X_te, y_tr, y_te = train_test_split(X1, y1, test_size=0.2, random_state=10, stratify=y1)

# Train the model
model1 = LogisticRegression(solver='liblinear', random_state=7)
model1.fit(X_tr, y_tr)
y_pred = model1.predict(X_te)

m1_acc = accuracy_score(y_te, y_pred)

# Plain English: the model correctly predicts cashless vs cash X% of the time
# Technical: accuracy = (true positives + true negatives) / total predictions
print(f"Model 1 — Cashless Payment Prediction")
print(f"  Result: {m1_acc * 100:.1f}% of predictions were correct")
print(f"  Technical: Accuracy = {m1_acc:.3f}\n")

# Confusion matrix — shows exactly where the model got it right and wrong
# Top-left: correctly predicted cashless | Top-right: said cashless but was cash
# Bottom-left: said cash but was cashless | Bottom-right: correctly predicted cash
cm = confusion_matrix(y_te, y_pred, labels=['yes', 'no'])
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Cashless', 'Cash']).plot()
plt.title('Model 1: Cashless vs Cash — How often was the model right?')
plt.show()

# Technical: precision, recall, F1-score per class
print(classification_report(y_te, y_pred, target_names=['Cashless', 'Cash']))

In [ ]:
# Which factors matter most for predicting cashless payments?
# Plain English: we remove features one by one and see which ones the model misses most
# Technical: Recursive Feature Elimination (RFE) — selects top 3 features

rfe = RFE(model1, n_features_to_select=3)
rfe.fit(X1, y1)
print("Top 3 factors that most influence whether a customer pays cashless:")
for name in X1.columns[rfe.support_]:
    print(f"  → {name}")

In [ ]:
# Can we improve the model by trying different settings?
# Plain English: we test 5 different "strictness" levels and pick the best one
# Technical: GridSearchCV tunes the regularisation parameter C with 5-fold cross-validation

grid = GridSearchCV(LogisticRegression(solver='liblinear'), {'C': [0.01, 0.1, 1, 10, 20]}, cv=5)
grid.fit(X_tr, y_tr)
print(f"Best setting found: C={grid.best_params_['C']}")
print(f"Cross-validation accuracy: {grid.best_score_*100:.1f}%")

best = grid.best_estimator_
best.fit(X_tr, y_tr)
print(f"Final accuracy after tuning: {accuracy_score(y_te, best.predict(X_te))*100:.1f}%")

---
# Model 2: Vendor Daily Revenue Forecast

**Business question:** How much revenue will a vendor earn on a given day?

**Who benefits:** Vendors can plan how much stock to prepare and how many staff to schedule.

**How it works (plain English):** We look at past days and see what drove higher or lower revenue — was it the day of week? Number of orders? Weekend? The model finds the pattern and predicts future revenue.

**Technical:** Linear Regression. Target variable is continuous (dollar amount). Evaluated with R² (proportion of variance explained, 0-1) and MAE (average prediction error in dollars).

In [ ]:
# Aggregate to vendor-day level: one row = one vendor's total for one day
vd = txn.groupby(['vendor_name', 'vendor_category', 'date', 'is_weekend']).agg(
    revenue=('amount', 'sum'),
    num_transactions=('transaction_id', 'count'),
    avg_order_value=('amount', 'mean'),
).reset_index()
vd['day_of_week'] = pd.to_datetime(vd['date']).dt.dayofweek

X2 = vd[['num_transactions', 'avg_order_value', 'is_weekend', 'day_of_week']].copy()
y2 = vd['revenue']
X2, _ = preprocess(X2)

X_tr, X_te, y_tr, y_te = train_test_split(X2, y2, test_size=0.2, random_state=10)
model2 = LinearRegression()
model2.fit(X_tr, y_tr)
y_pred = model2.predict(X_te)

m2_r2 = r2_score(y_te, y_pred)
m2_mae = mean_absolute_error(y_te, y_pred)

# Plain English: the model explains X% of why revenue goes up or down,
# and predictions are typically off by $X
# Technical: R² = coefficient of determination, MAE = mean absolute error
print(f"Model 2 — Vendor Daily Revenue Forecast")
print(f"  Result: {m2_r2*100:.1f}% accurate — predictions off by ~${m2_mae:.2f} on average")
print(f"  Technical: R² = {m2_r2:.3f}, MAE = ${m2_mae:.2f}")

---
# Model 3: Demand Forecast by Hour

**Business question:** How many orders will the platform receive in each hour of the evening?

**Who benefits:** Admin can plan event logistics. Vendors know when to prepare more food.

**How it works (plain English):** Orders follow a bell curve — low at 6PM, peak around 9PM, drop by 11PM. A straight line can't capture this curve, so we add "hour squared" as a feature which lets the model bend to fit the peak.

**Technical:** Linear Regression with polynomial feature (hour²) to model the quadratic relationship between hour and order volume. Additional features: day_of_week, is_weekend, unique_vendors, avg_amount.

In [ ]:
# Aggregate to date-hour level
hourly = txn.groupby(['date', 'hour']).agg(
    orders=('transaction_id', 'count'),
    revenue=('amount', 'sum'),
    avg_amount=('amount', 'mean'),
    unique_vendors=('vendor_name', 'nunique'),
).reset_index()
hourly['date'] = pd.to_datetime(hourly['date'])
hourly['day_of_week'] = hourly['date'].dt.dayofweek
hourly['is_weekend'] = (hourly['day_of_week'] >= 5).astype(int)

# hour² lets the model capture the peak-hour bell curve
# Technical: polynomial feature engineering
hourly['hour_sq'] = hourly['hour'] ** 2

X3 = hourly[['hour', 'hour_sq', 'day_of_week', 'is_weekend', 'unique_vendors', 'avg_amount']].copy()
y3 = hourly['orders']
X3, _ = preprocess(X3)

X_tr, X_te, y_tr, y_te = train_test_split(X3, y3, test_size=0.2, random_state=10)
model3 = LinearRegression()
model3.fit(X_tr, y_tr)
y_pred = model3.predict(X_te)

m3_r2 = r2_score(y_te, y_pred)
m3_mae = mean_absolute_error(y_te, y_pred)

# Plain English: the model predicts hourly orders with X% accuracy,
# typically off by about X orders
# Technical: R² = 0.944 means the model explains 94.4% of hourly order variation
print(f"Model 3 — Demand Forecast by Hour")
print(f"  Result: {m3_r2*100:.1f}% accurate — predictions off by ~{m3_mae:.1f} orders")
print(f"  Technical: R² = {m3_r2:.3f}, MAE = {m3_mae:.2f}")

---
# Model 4: Review Sentiment Classification

**Business question:** Is a customer review positive or negative?

**Who benefits:** Vendors can monitor customer satisfaction. Admin can rank vendors by how happy their customers are.

**How it works (plain English):** We check if the review contains positive words ("amazing", "delicious", "recommend") or negative words ("disappointed", "overpriced", "stale"). More positive words = positive review.

**Technical:** Keyword-based NLP classification. Ground truth labels derived from star ratings (4-5 = POSITIVE, 1-2 = NEGATIVE). Also demonstrated with Hugging Face distilbert in Notebook 2.

In [ ]:
# Keyword-based sentiment scoring
positive_words = {'amazing', 'love', 'great', 'best', 'delicious', 'incredible',
                  'outstanding', 'perfect', 'enjoyed', 'recommend', 'must-try',
                  'friendly', 'fresh', 'tasty', 'excellent'}
negative_words = {'disappointed', 'overpriced', 'stale', 'slow', 'cold', 'worst',
                  'terrible', 'awful', 'bad', 'waste'}

def predict_sentiment(text):
    words = set(text.lower().split())
    pos = len(words & positive_words)
    neg = len(words & negative_words)
    return 'POSITIVE' if pos > neg else 'NEGATIVE'

reviews['predicted'] = reviews['content'].apply(predict_sentiment)

m4_acc = accuracy_score(reviews['sentiment'], reviews['predicted'])

# Plain English: the model correctly identifies X% of reviews as positive or negative
# Technical: accuracy = correct predictions / total predictions
print(f"Model 4 — Review Sentiment Classification")
print(f"  Result: correctly classifies {m4_acc*100:.1f}% of {len(reviews)} reviews")
print(f"  Technical: Accuracy = {m4_acc:.3f}\n")

# Per-week breakdown — shows consistency over time
reviews['week'] = reviews.index // (len(reviews) // 4)
for w in range(4):
    wdf = reviews[reviews['week'] == w]
    wacc = accuracy_score(wdf['sentiment'], wdf['predicted'])
    print(f"  Week {w+1}: {wacc*100:.1f}% correct on {len(wdf)} reviews")

# Confusion matrix
cm = confusion_matrix(reviews['sentiment'], reviews['predicted'], labels=['POSITIVE', 'NEGATIVE'])
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Positive', 'Negative']).plot()
plt.title('Model 4: Review Classification — How often was the model right?')
plt.show()

# Technical: precision, recall, F1 per class
print(classification_report(reviews['sentiment'], reviews['predicted'], target_names=['Positive', 'Negative']))

---
# Model 5: Vendor Total Revenue Forecast

**Business question:** Given a vendor's order count, average order value, days active, and category — what will their total revenue be?

**Who benefits:** Admin can rank vendors by predicted performance and make better stall allocation decisions.

**How it works (plain English):** We look at all vendors' past performance and find the relationship between their activity (orders, days, order size) and their total earnings. Then we predict what a vendor should earn based on those patterns.

**Technical:** Linear Regression on aggregated vendor-level features. High R² expected since total_orders × avg_order ≈ total_revenue by construction — but the model also captures category and weekend effects.

In [ ]:
# Aggregate to vendor totals
vtotals = txn.groupby(['vendor_name', 'vendor_category']).agg(
    total_revenue=('amount', 'sum'),
    total_orders=('transaction_id', 'count'),
    avg_order=('amount', 'mean'),
    num_days=('date', 'nunique'),
    pct_weekend=('is_weekend', 'mean'),
).reset_index()

X5 = vtotals[['total_orders', 'avg_order', 'num_days', 'pct_weekend', 'vendor_category']].copy()
y5 = vtotals['total_revenue']
X5, _ = preprocess(X5)

X_tr, X_te, y_tr, y_te = train_test_split(X5, y5, test_size=0.2, random_state=10)
model5 = LinearRegression()
model5.fit(X_tr, y_tr)
y_pred = model5.predict(X_te)

m5_r2 = r2_score(y_te, y_pred)
m5_mae = mean_absolute_error(y_te, y_pred)

# Plain English: the model predicts vendor total revenue with X% accuracy,
# typically off by ~$X
# Technical: R² close to 1.0 because features are strongly correlated with target
print(f"Model 5 — Vendor Total Revenue Forecast")
print(f"  Result: {m5_r2*100:.1f}% accurate — predictions off by ~${m5_mae:.2f}")
print(f"  Technical: R² = {m5_r2:.3f}, MAE = ${m5_mae:.2f}")

---
# Summary: All 5 Models

In [ ]:
print("=" * 75)
print("  ALL MODELS — RESULTS SUMMARY")
print("=" * 75)

print(f"\n  Model 1: Cashless Payment Prediction")
print(f"    What it does:  Predicts if a customer will pay cashless or cash")
print(f"    Result:        {m1_acc*100:.1f}% of predictions correct")
print(f"    Technical:     Logistic Regression | Accuracy = {m1_acc:.3f}")

print(f"\n  Model 2: Vendor Daily Revenue")
print(f"    What it does:  Predicts how much a vendor will earn on a given day")
print(f"    Result:        {m2_r2*100:.1f}% accurate, off by ~${m2_mae:.2f}")
print(f"    Technical:     Linear Regression | R² = {m2_r2:.3f}, MAE = ${m2_mae:.2f}")

print(f"\n  Model 3: Hourly Demand Forecast")
print(f"    What it does:  Predicts order volume per hour")
print(f"    Result:        {m3_r2*100:.1f}% accurate, off by ~{m3_mae:.1f} orders")
print(f"    Technical:     Linear Regression + hour² | R² = {m3_r2:.3f}, MAE = {m3_mae:.2f}")

print(f"\n  Model 4: Review Sentiment")
print(f"    What it does:  Classifies customer reviews as positive or negative")
print(f"    Result:        {m4_acc*100:.1f}% of reviews correctly classified")
print(f"    Technical:     Keyword NLP | Accuracy = {m4_acc:.3f}")

print(f"\n  Model 5: Vendor Total Revenue")
print(f"    What it does:  Predicts a vendor's total earnings")
print(f"    Result:        {m5_r2*100:.1f}% accurate, off by ~${m5_mae:.2f}")
print(f"    Technical:     Linear Regression | R² = {m5_r2:.3f}, MAE = ${m5_mae:.2f}")

print("\n" + "=" * 75)
print("  Each model is independent with its own data, features, and metric.")
print("=" * 75)